# Appendix C — Budget saturation and discovery allocation

Use saved cold-start predictions and AL trajectories to inspect why the
confidence tail can exhaust a fixed budget at larger volume. Analyze the
allocation trade-off on the same pool; this is sensitivity analysis, not an
independent confirmation or a universal optimum search.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, tempfile
from pathlib import Path
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'mpl_cache'))
raiz = Path.cwd()
while not (raiz / 'config.py').exists() and raiz != raiz.parent:
    raiz = raiz.parent
os.chdir(raiz); sys.path.insert(0, str(raiz))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
import config
from src.models import classification
PROC = config.DIR_PROCESSED
plt.rcParams.update({'figure.dpi':120,'axes.grid':True,'grid.alpha':.25,
                     'axes.spines.top':False,'axes.spines.right':False})


dc   = pd.read_csv(PROC / 'dos_canales_batch99_10seeds.csv')
tri  = pd.read_csv(PROC / 'triaje_batch99_10seeds.csv')
fino = pd.read_csv(PROC / 'dos_canales_batch99_fino_10s.csv')
fin  = dc.sort_values('iter').groupby(['frac_desc', 'semilla']).tail(1)
ftr  = tri.sort_values('iter').groupby(['modo', 'semilla']).tail(1)
print('two channels:', sorted(fin.frac_desc.unique()), '· seeds', fin.semilla.nunique())
print('fine sweep:', sorted(fino.frac_desc.unique()), '· seeds', fino.semilla.nunique())


## 1. Candidate counts and the fixed budget


In [ ]:

pr = pd.read_csv(PROC / 'predicciones_test_batch99.csv')
n_pool = len(pr); cand_pool = int((pr.confianza < 0.30).sum()); tasa_pool = cand_pool / n_pool


from src.models.subsampling import load_validation
_, validated_summary, _, validation_manifest = load_validation(PROC)
n_b4 = int(validation_manifest['n_full'])
candidate_rate_b4 = float(validated_summary.iloc[0]['batch_completo']) / 100
cand_b4 = int(round(n_b4 * candidate_rate_b4))
BUDGET = 1000; n_cruce = BUDGET / tasa_pool
print(f"Full batch 4: {cand_b4:5d} / {n_b4:,} ({100*cand_b4/n_b4:.3f} %, Appendix A) -> {'Below budget' if cand_b4<BUDGET else 'Saturated'}")
print(f"Eleven-batch pool: {cand_pool:5d} / {n_pool:,} ({100*tasa_pool:.3f} %) -> {'Below budget' if cand_pool<BUDGET else 'Saturated'}")
print(f'Crossover at the conservative pool rate ≈ {n_cruce/1000:.0f}k objects')

fig, ax = plt.subplots(figsize=(8.2, 4.8))
xg = np.linspace(0, 1.05*n_pool, 300)
ax.plot(xg, tasa_pool*xg, lw=2.3, color='C0', label=f'Projection using the pool rate {100*tasa_pool:.3f} %)')
ax.axhline(BUDGET, color='C3', ls='--', lw=2, label=f'budget = {BUDGET:,}')
ax.axvline(n_cruce, color='.35', ls=':', lw=1.8, label=f'Crossover ≈ {n_cruce/1000:.0f}k objects')
ax.scatter([n_b4], [cand_b4], s=85, color='C2', zorder=5, edgecolor='k',
           label=f'batch 4 full: {cand_b4} / {n_b4//1000}k (Appendix A)')
ax.scatter([n_pool], [cand_pool], s=85, color='C0', zorder=5, edgecolor='k',
           label=f'pool 11 batches: {cand_pool:,} / {n_pool//1000}k')
ax.set(xlabel='Number of processed objects', ylabel='Candidates with confidence < 0.30',
       title='Saturation under a fixed budget as volume grows')
ax.set_xlim(0, 1.05*n_pool); ax.set_ylim(0, max(cand_pool, cand_b4, BUDGET)*1.18)
ax.legend(fontsize=8.5, loc='upper left')
fig.tight_layout()
fig.savefig(config.FIGURE_DIR / 'mecanismo_saturacion_presupuesto.png', dpi=150, bbox_inches='tight')
plt.show()


### Saturation interpretation

Compare candidate counts with the 1,000-query budget. The verified full-batch count is read from the subsampling results generated
with the canonical final model. Pool volume and accumulated
heterogeneity both change between the reference batch and the combined pool.


## 2. Discovery-allocation sensitivity


### 2.1. Classification and discovery together


In [ ]:
def summary_table(df, by):
    g = df.groupby(by).agg(MF1=('MF1','mean'), MF1_sd=('MF1','std'),
                           descub=('n_descubiertas','mean'), descub_sd=('n_descubiertas','std'),
                           n=('MF1','size'))
    g['MF1_se'] = g.MF1_sd/np.sqrt(g.n); g['descub_se'] = g.descub_sd/np.sqrt(g.n)
    return g

R = summary_table(fin, 'frac_desc'); nseed = int(R.n.iloc[0])
ref = ftr.groupby('modo').agg(MF1=('MF1','mean'), descub=('n_descubiertas','mean'))
print(f'{nseed} seeds · mean ± standard error')
print(R[['MF1','MF1_se','descub','descub_se']].round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.7))
for a, (col, se, ylab, ttl) in zip(ax, [('MF1','MF1_se','Macro-F1 (known classes)','Classification'),
                                        ('descub','descub_se','Discovered anomalies','Anomaly discovery')]):
    a.axvspan(0.3, 0.6, color='C2', alpha=.10, label='Stable region (0.3–0.6)')
    a.errorbar(R.index, R[col], yerr=R[se], marker='o', color='C0', capsize=3, lw=2, label='two channels')
    for m, c, l in [('triage', '.45', 'triage'), ('random', 'C7', 'random')]:
        if m in ref.index:
            a.axhline(ref.loc[m, col], ls='--', color=c, lw=1.3, label=l)
    a.set(xlabel='frac_desc', ylabel=ylab, title=ttl); a.legend(fontsize=8)
fig.suptitle(f'Allocation robustness · combined pool ({nseed} seeds)', y=1.02)
fig.tight_layout()
fig.savefig(config.FIGURE_DIR / 'curva_frac_desc_batch99.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.2. Extreme allocations sacrifice one objective


In [ ]:
def pareado(col, base, target):
    P = fin.pivot_table(index='semilla', columns='frac_desc', values=col)
    x, y = P[base].values, P[target].values
    _, p = wilcoxon(x, y); d = y - x
    return d.mean(), d.std(ddof=1), p

print('Tests pareados by seed (Wilcoxon):\n')
print(f"{'Contrast':<26}{'ΔMF1':>9}{'p(MF1)':>9}   {'Δfound':>9}{'p(found)':>9}")
for base, tgt, lab in [(0.0, 0.6, '0.6 vs 0.0 (clas. pura)'), (1.0, 0.6, '0.6 vs 1.0 (desc. pura)'),
                       (0.0, 0.3, '0.3 vs 0.0'), (1.0, 0.3, '0.3 vs 1.0')]:
    dM, _, pM = pareado('MF1', base, tgt); dD, _, pD = pareado('n_descubiertas', base, tgt)
    print(f'{lab:<26}{dM:>+9.4f}{pM:>9.3f}   {dD:>+9.1f}{pD:>9.3f}')

sd = fin.groupby('frac_desc')['n_descubiertas'].std().round(1)
print('\nDiscovery standard deviation across seeds by allocation:', dict(sd))


### 2.3. Compare allocations 0.3 and 0.6


In [ ]:
dM, sM, pM = pareado('MF1', 0.3, 0.6)
dD, sD, pD = pareado('n_descubiertas', 0.3, 0.6)
print('Allocation 0.6 versus 0.3 (paired seed differences):')
print(f'  MF1          : Δ = {dM:+.4f}  (sd {sM:.4f})  Wilcoxon p = {pM:.3f}')
print(f'  descubiertas : Δ = {dD:+.2f}    (sd {sD:.2f})    Wilcoxon p = {pD:.3f}')
print('\nNeither objective distinguishes allocations 0.3 and 0.6 conclusively in this experiment.')


### 2.4. Classification–discovery trade-off


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))

ax[0].errorbar(R['descub'], R['MF1'], xerr=R['descub_se'], yerr=R['MF1_se'], fmt='none', ecolor='C0', alpha=.5)
for fr in R.index:
    ax[0].scatter(R.loc[fr,'descub'], R.loc[fr,'MF1'], s=75, color='C0', zorder=5)
    ax[0].annotate(f'frac={fr}', (R.loc[fr,'descub'], R.loc[fr,'MF1']),
                   fontsize=9, textcoords='offset points', xytext=(6, 4))
for m, c in [('triage', '.45'), ('random', 'C7')]:
    if m in ref.index:
        ax[0].scatter(ref.loc[m,'descub'], ref.loc[m,'MF1'], s=65, marker='s', color=c, label=m, zorder=5)
ax[0].set(xlabel='Discovered anomalies', ylabel='MF1', title='Classification–discovery trade-off')
ax[0].legend(fontsize=8)

nM = (R['MF1']-R['MF1'].min())/(R['MF1'].max()-R['MF1'].min())
nD = (R['descub']-R['descub'].min())/(R['descub'].max()-R['descub'].min())
bal = np.minimum(nM, nD)
ax[1].bar([str(f) for f in R.index], bal.values, color=['C7','C0','C0','C7'])
for i, v in enumerate(bal.values):
    ax[1].text(i, v+0.02, f'{v:.2f}', ha='center', fontsize=9)
ax[1].set(xlabel='frac_desc', ylabel='Trade-off index (normalized maximin)',
          title='Equilibrio entre ambos objetivos', ylim=(0, 1.12))
fig.tight_layout()
fig.savefig(config.FIGURE_DIR / 'compromiso_frac_desc_batch99.png', dpi=150, bbox_inches='tight')
plt.show()
print('Trade-off index by discovery allocation:', {float(f): round(float(v), 2) for f, v in bal.items()})


### 2.5. Finer allocation sweep


In [ ]:
finf = fino.sort_values('iter').groupby(['frac_desc', 'semilla']).tail(1)
Rf = finf.groupby('frac_desc').agg(MF1=('MF1','mean'), MF1_sd=('MF1','std'),
                                   descub=('n_descubiertas','mean'), descub_sd=('n_descubiertas','std'),
                                   n=('MF1','size'))
nf = int(Rf.n.iloc[0])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].errorbar(Rf.index, Rf.MF1, yerr=Rf.MF1_sd/np.sqrt(nf), marker='o', capsize=3, color='C0')
ax[0].axvspan(0.3, 0.6, color='C2', alpha=.10); ax[0].set(xlabel='frac_desc', ylabel='MF1', title=f'MF1 (fine sweep, {nf} seeds)')
ax[1].errorbar(Rf.index, Rf.descub, yerr=Rf.descub_sd/np.sqrt(nf), marker='o', capsize=3, color='C3')
ax[1].axvspan(0.3, 0.6, color='C2', alpha=.10); ax[1].set(xlabel='frac_desc', ylabel='Discovered anomalies', title='Anomaly discovery (fine sweep)')
fig.suptitle('Fine allocation sweep (0.1–0.8): stable Macro-F1, increasing discovery', y=1.02)
fig.tight_layout()
fig.savefig(config.FIGURE_DIR / 'curva_frac_desc_fino_batch99.png', dpi=150, bbox_inches='tight')
plt.show()


## Interpretation limits

Intermediate allocations provide a useful compromise in this experiment.
Failure to detect a difference between 0.3 and 0.6 does not prove equivalence.
The choice depends on anomaly prevalence, query budget and scientific priorities;
the sensitivity sweep uses the same data as the main large-pool comparison.
